# Multi-Model Seasonal Forecast — Stage 8 Notebook

## KENYA MAM 2026 — C3S Multi-Model Ensemble

**Purpose:** Combine outputs from all individual model pipelines into a single
multi-model bulletin using `bulletin_multimodel_v1.py`.

**Models included:**
- ECMWF SEAS5
- UK Met Office GloSea6
- Météo-France System 8
- DWD GCFS2.1
- CMCC-SPS4
- NCEP CFSv2
- ECCC CanSIPS
- BOM ACCESS-S2

**Prerequisites:** Each model pipeline notebook must have been run and its
`.nc` output files saved to `OUT_DIR` before running this notebook.

---

### Workflow
1. Load shared CHIRPS reference arrays
2. Load each model's saved detection + calibration outputs from disk
3. Build the `MODELS` dict
4. Set multi-model metadata overrides
5. Execute `bulletin_multimodel_v1.py` → generates PNG + PDF per site


## Imports & Configuration


In [1]:
import os
import numpy  as np
import xarray as xr
import datetime

print('Imports OK')


Imports OK


## Configuration — Paths and Shared Parameters

Set all paths here. `CHIRPS_DIR` holds the shared CHIRPS arrays.
Each model has its own output directory produced by its pipeline notebook.


In [ ]:
# ── Base directory ────────────────────────────────────────────────────────
BASE_DIR   = r'C:\Users\yonas\Documents\ILRI\onset-kenya'

# ── Shared CHIRPS reference (from SEAS5 pipeline Stage 3A/4A outputs) ────
CHIRPS_DIR = os.path.join(BASE_DIR, 'outputs_dunning_v3')

# ── Per-model output directories ──────────────────────────────────────────
MODEL_DIRS = {
    'ECMWF SEAS5'       : os.path.join(BASE_DIR, 'outputs_dunning_v3'),
    'UKMO GloSea6'      : os.path.join(BASE_DIR, 'outputs_ukmo_v3'),
    'Meteo-France Sys8' : os.path.join(BASE_DIR, 'outputs_mf_v3'),
    'DWD GCFS2.1'       : os.path.join(BASE_DIR, 'outputs_dwd_v3'),
    'CMCC-SPS4'         : os.path.join(BASE_DIR, 'outputs_cmcc_v3'),
    'NCEP CFSv2'        : os.path.join(BASE_DIR, 'outputs_ncep_v3'),
    'ECCC CanSIPS'      : os.path.join(BASE_DIR, 'outputs_eccc_v3'),
    'BOM ACCESS-S2'     : os.path.join(BASE_DIR, 'outputs_bom_v3'),
}

# ── Multi-model bulletin output directory ─────────────────────────────────
MM_OUT_DIR   = os.path.join(BASE_DIR, 'outputs_multimodel')
BULLETIN_DIR = os.path.join(MM_OUT_DIR, 'bulletins')
os.makedirs(BULLETIN_DIR, exist_ok=True)

# ── Shared pipeline constants ─────────────────────────────────────────────
OP_YEAR       = 2026
WIN_DOY_START = 32
WIN_DOY_END   = 213

# ── Per-model colours ─────────────────────────────────────────────────────
MODEL_COLORS = {
    'ECMWF SEAS5'       : '#1B5EA6',   # blue
    'UKMO GloSea6'      : '#C0392B',   # red
    'Meteo-France Sys8' : '#1E6B45',   # green
    'DWD GCFS2.1'       : '#8B4513',   # brown
    'CMCC-SPS4'         : '#7B0EA6',   # purple
    'NCEP CFSv2'        : '#C9920A',   # gold
    'ECCC CanSIPS'      : '#0E7490',   # teal
    'BOM ACCESS-S2'     : '#E63946',   # crimson
}

# ── Bulletin sites ────────────────────────────────────────────────────────
SITES = [
    {'site_name': 'El Karama Sahiwals Farm',
     'lat': -2.3871,  'lon': 37.4851},
    {'site_name': 'Genco LTD Maralal Samburu Farm',
     'lat':  0.9273,  'lon': 36.5690},
    {'site_name': 'Genco LTD Tana River Farm',
     'lat': -2.21314, 'lon': 40.0517},
    {'site_name': 'KALRO Kiboko, Makueni Farm',
     'lat': -2.21046, 'lon': 37.7190},
    {'site_name': 'Kapiti Research Station Farm',
     'lat': -1.63209, 'lon': 37.1479},
    {'site_name': 'LiveMo LTD, Memerush, Kajiado Farm',
     'lat': -2.38726, 'lon': 37.4850},
]

print('Configuration loaded:')
print(f'  Models          : {len(MODEL_DIRS)}')
print(f'  Output dir      : {BULLETIN_DIR}')
print(f'  OP year         : {OP_YEAR}')
for name, path in MODEL_DIRS.items():
    exists = os.path.isdir(path)
    print(f'  {name:<22}: {"OK" if exists else "MISSING"} -- {path}')


Configuration loaded:
  Models          : 8
  Output dir      : C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_multimodel\bulletins
  OP year         : 2026
  ECMWF SEAS5           : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_seas5_v3
  UKMO GloSea6          : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_ukmo_v3
  Meteo-France Sys8     : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_mf_v3
  DWD GCFS2.1           : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_dwd_v3
  CMCC-SPS4             : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_cmcc_v3
  NCEP CFSv2            : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_ncep_v3
  ECCC CanSIPS          : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_eccc_v3
  BOM ACCESS-S2         : OK -- C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_bom_v3


## Stage 1 — Load Shared CHIRPS Reference Arrays

These are identical for all models — loaded once from the SEAS5
pipeline outputs (Stages 3A, 4A).


In [3]:
# ── CHIRPS reference arrays ────────────────────────────────────────────────

def _load(path, var=None):
    """Load first variable from netCDF as float32. Suppresses timedelta decode."""
    ds  = xr.open_dataset(path, decode_timedelta=False)
    key = var if var else list(ds.data_vars)[0]
    arr = ds[key].values.astype(np.float32)
    ds.close()
    return arr

def _load_any(dir_, *candidates, var=None):
    """Try each filename in dir_ in order; raise if none found."""
    for name in candidates:
        p = os.path.join(dir_, name)
        if os.path.exists(p):
            return _load(p, var=var)
    raise FileNotFoundError(
        "None found:\n" + "\n".join(f"  {os.path.join(dir_,n)}" for n in candidates))

def _load_any2(d1, d2, *names):
    """Try names in d1 first, then d2."""
    for d in [d1, d2]:
        for n in names:
            p = os.path.join(d, n)
            if os.path.exists(p):
                return _load(p)
    tried = [os.path.join(d,n) for d in [d1,d2] for n in names]
    raise FileNotFoundError("None found:\n" + "\n".join(f"  {t}" for t in tried))

print("Loading shared CHIRPS reference arrays ...")
print(f"  From: {CHIRPS_DIR}")

# Detection arrays
chirps_onset_doy = _load_any(CHIRPS_DIR,
    "CHIRPS_onset_doy_1981_2025.nc","CHIRPS_onset_doy_all_years.nc","CHIRPS_onset_doy.nc")
chirps_cessation_doy = _load_any(CHIRPS_DIR,
    "CHIRPS_cessation_doy_1981_2025.nc","CHIRPS_cessation_doy_all_years.nc","CHIRPS_cessation_doy.nc")
chirps_lgp_days = _load_any(CHIRPS_DIR,
    "CHIRPS_lgp_days_1981_2025.nc","CHIRPS_lgp_days_all_years.nc","CHIRPS_lgp_days.nc")

# LGP nanosecond guard
_lgp_abs = np.nanmax(np.abs(chirps_lgp_days))
if _lgp_abs > 1e6:
    print(f"  WARNING: LGP max={_lgp_abs:.2e} looks like nanoseconds; converting to days ...")
    chirps_lgp_days = (chirps_lgp_days / 86400e9).astype(np.float32)
chirps_lgp_days = np.where((chirps_lgp_days > 0) & (chirps_lgp_days <= 366),
                            chirps_lgp_days, np.nan).astype(np.float32)

# Dunning climatology
C_clim = _load_any(CHIRPS_DIR, "chirps_C_clim.nc", "C_clim.nc")
Q_bar  = _load_any(CHIRPS_DIR, "chirps_Q_bar.nc",  "Q_bar.nc")
d_s    = _load_any(CHIRPS_DIR, "chirps_d_s.nc",    "d_s.nc")
d_e    = _load_any(CHIRPS_DIR, "chirps_d_e.nc",    "d_e.nc")

# Tercile boundaries — try calibration_params subdir, then root
_cal_dir = os.path.join(CHIRPS_DIR, "calibration_params")
t33_onset = _load_any2(_cal_dir, CHIRPS_DIR, "t33_onset_doy.nc", "t33_onset.nc")
t67_onset = _load_any2(_cal_dir, CHIRPS_DIR, "t67_onset_doy.nc", "t67_onset.nc")
t33_cess  = _load_any2(_cal_dir, CHIRPS_DIR, "t33_cessation_doy.nc","t33_cess.nc","t33_cessation.nc")
t67_cess  = _load_any2(_cal_dir, CHIRPS_DIR, "t67_cessation_doy.nc","t67_cess.nc","t67_cessation.nc")
t33_lgp   = _load_any2(_cal_dir, CHIRPS_DIR, "t33_lgp_days.nc",  "t33_lgp.nc")
t67_lgp   = _load_any2(_cal_dir, CHIRPS_DIR, "t67_lgp_days.nc",  "t67_lgp.nc")

# Grid coordinates + land mask
_ref_ds = None
for _c in ["SEAS5_bc_daily_all_years.nc","SEAS5_BC_daily_all_years.nc"]:
    _p2 = os.path.join(CHIRPS_DIR, _c)
    if os.path.exists(_p2):
        _ref_ds = xr.open_dataset(_p2, decode_timedelta=False); break
if _ref_ds is None:
    for _c in ["CHIRPS_onset_doy_1981_2025.nc","CHIRPS_onset_doy.nc"]:
        _p2 = os.path.join(CHIRPS_DIR, _c)
        if os.path.exists(_p2):
            _ref_ds = xr.open_dataset(_p2, decode_timedelta=False); break
for _lname in ["lat","latitude","y"]:
    if _lname in _ref_ds.coords: target_lat = _ref_ds[_lname].values.astype(np.float64); break
for _oname in ["lon","longitude","x"]:
    if _oname in _ref_ds.coords: target_lon = _ref_ds[_oname].values.astype(np.float64); break
_ref_ds.close()
n_lat, n_lon = len(target_lat), len(target_lon)
lm = np.isfinite(chirps_onset_doy).any(axis=0)

# chirps_years from file coordinate (not hardcoded)
_onset_path = None
for _c in ["CHIRPS_onset_doy_1981_2025.nc","CHIRPS_onset_doy_all_years.nc","CHIRPS_onset_doy.nc"]:
    _tp = os.path.join(CHIRPS_DIR, _c)
    if os.path.exists(_tp): _onset_path = _tp; break
_ds2 = xr.open_dataset(_onset_path, decode_timedelta=False)
if "year" in _ds2.coords:
    chirps_years = _ds2["year"].values.astype(int)
elif "time" in _ds2.coords:
    chirps_years = np.array([int(str(t)[:4]) for t in _ds2["time"].values])
else:
    _nyr = int(_ds2[list(_ds2.data_vars)[0]].shape[0])
    chirps_years = np.arange(1981, 1981 + _nyr)
    print(f"  WARNING: no year coord; inferred {chirps_years[0]}-{chirps_years[-1]}")
_ds2.close()

if len(chirps_years) != chirps_onset_doy.shape[0]:
    raise RuntimeError(
        f"chirps_years len={len(chirps_years)} != onset array dim={chirps_onset_doy.shape[0]}")

CAL_YEARS  = np.arange(1981, 2017)
cal_idx    = np.where(np.isin(chirps_years, CAL_YEARS))[0]
cal_mask_c = np.isin(chirps_years, CAL_YEARS)
if cal_mask_c.sum() == 0:
    raise RuntimeError(f"No CAL years found in chirps_years={chirps_years[:4]}...")

chirps_clim = {
    "onset"    : np.where(lm, np.nanmean(chirps_onset_doy[cal_mask_c],     axis=0), np.nan).astype(np.float32),
    "cessation": np.where(lm, np.nanmean(chirps_cessation_doy[cal_mask_c], axis=0), np.nan).astype(np.float32),
    "lgp"      : np.where(lm, np.nanmean(chirps_lgp_days[cal_mask_c],      axis=0), np.nan).astype(np.float32),
}
print(f"  Grid         : {n_lat} x {n_lon}  ({int(lm.sum())} land pixels)")
print(f"  chirps_years : {chirps_years[0]}-{chirps_years[-1]}  ({len(chirps_years)} yrs)")
print(f"  CAL match    : {int(cal_mask_c.sum())} years")
print(f"  Onset mean   : DOY {np.nanmean(chirps_clim["onset"][lm]):.1f}")
print(f"  Cess mean    : DOY {np.nanmean(chirps_clim["cessation"][lm]):.1f}")
print(f"  LGP mean     : {np.nanmean(chirps_clim["lgp"][lm]):.1f} days")
print(f"  C_clim shape : {C_clim.shape}")
print("CHIRPS arrays loaded OK")


Loading shared CHIRPS reference arrays ...


C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\423157748.py:7: FutureWarning: In a future version, xarray will not decode the variable '__xarray_dataarray_variable__' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(path)
C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\423157748.py:7: FutureWarning: In a future version, xarray will not decode the variable '__xarray_dataarray_variable__' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'd

  Grid            : 42 lat x 34 lon  (833 land pixels)
  CHIRPS years    : 1981-2025
  CAL indices     : 0-35  (36 yrs)
  CHIRPS CAL onset: DOY 85.7
  C_clim shape    : (182, 42, 34)
Shared CHIRPS arrays loaded OK


C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\423157748.py:61: RuntimeWarning: Mean of empty slice
  np.nanmean(chirps_onset_doy[cal_mask_c], axis=0),
C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\423157748.py:64: RuntimeWarning: Mean of empty slice
  np.nanmean(chirps_cessation_doy[cal_mask_c], axis=0),


## Stage 2 — Load Per-Model Pipeline Outputs

Each model's pipeline notebook saves the following `.nc` files:

| File | Shape | Description |
|---|---|---|
| `{MODEL}_onset_doy_all_years.nc` | (n_mem, n_yrs, n_lat, n_lon) | Dunning onset DOY |
| `{MODEL}_cessation_doy_all_years.nc` | (n_mem, n_yrs, n_lat, n_lon) | Cessation DOY |
| `{MODEL}_lgp_days_all_years.nc` | (n_mem, n_yrs, n_lat, n_lon) | LGP days |
| `{MODEL}_bc_daily_all_years.nc` | (n_mem, n_yrs, 182, n_lat, n_lon) | QM-BC daily mm/day |
| `probs_damped_onset.nc` | (3, n_yrs, n_lat, n_lon) | BN/NN/AN probabilities |
| `probs_damped_cessation.nc` | (3, n_yrs, n_lat, n_lon) | |
| `probs_damped_lgp.nc` | (3, n_yrs, n_lat, n_lon) | |
| `alpha_onset.nc` | (n_lat, n_lon) | Optimal pooling weight |
| `hitrate_cal_onset.nc` | (n_lat, n_lon) | CAL hit rate |
| `hitrate_val_onset.nc` | (n_lat, n_lon) | VAL hit rate |
| `rpss_val_onset.nc` | (n_lat, n_lon) | VAL RPSS |

> **Note:** The `bc_daily` array is large (~200 MB per model). If RAM is
> tight, set `LOAD_BC_DAILY = False` — the A(D) plume panel will be skipped
> and all other panels will still render.


In [4]:
LOAD_BC_DAILY = True   # set False to skip A(D) plume and save ~200 MB/model RAM

def _load_model(name, out_dir):
    """
    Load all pipeline output files for one model from out_dir.
    Uses glob-based auto-discovery for skill files so naming convention
    differences cannot cause silent NaN fills.
    """
    if not os.path.isdir(out_dir):
        print(f"  SKIP  {name}: dir not found -- {out_dir}")
        return None

    import glob as _glob

    # ── File prefix from directory name ───────────────────────────────────
    _prefix_map = {
        "dunning_v3":"SEAS5","bom_v3":"BOM","ukmo_v3":"UKMO","mf_v3":"MF",
        "dwd_v3":"DWD","cmcc_v3":"CMCC","ncep_v3":"NCEP","eccc_v3":"ECCC",
    }
    pfx = next((v for k,v in _prefix_map.items()
                if k in os.path.basename(out_dir)), "MODEL")

    def _p(fname): return os.path.join(out_dir, fname)

    def _safe_load(path, var=None):
        """Load netCDF or None if missing. decode_timedelta suppressed."""
        if not os.path.exists(path): return None
        ds  = xr.open_dataset(path, decode_timedelta=False)
        v   = list(ds.data_vars)[0] if var is None else var
        raw = ds[v].values; ds.close()
        arr = raw.astype(np.float32)
        if np.nanmax(np.abs(arr)) > 1e9:          # ns timedelta guard
            arr = (arr / 86400e9).astype(np.float32)
        return arr

    # ── Detection arrays ──────────────────────────────────────────────────
    def _try_load(base):
        for pat in [f"{pfx}_{base}_all_years.nc", f"{pfx}_{base}_1993_2026.nc",
                    f"{pfx}_{base}_1981_2026.nc",  f"{pfx}_{base}.nc"]:
            r = _safe_load(_p(pat))
            if r is not None: return r
        return None

    onset_doy = _try_load("onset_doy")
    cess_doy  = _try_load("cessation_doy")
    lgp_days  = _try_load("lgp_days")

    if onset_doy is None:
        print(f"  SKIP  {name}: onset_doy not found in {out_dir}")
        return None

    n_members = int(onset_doy.shape[0])
    n_years   = int(onset_doy.shape[1])

    # ── Year array + op_idx ───────────────────────────────────────────────
    _yfile = _p("model_years.nc")
    if os.path.exists(_yfile):
        model_years = xr.open_dataset(_yfile)["year"].values.astype(int)
    else:
        start = 1981 if pfx == "SEAS5" else 1993
        model_years = np.array([y for y in range(start, start+n_years-1)] + [OP_YEAR])
        if len(model_years) != n_years:
            model_years = np.arange(start, start+n_years)
    op_mask = (model_years == OP_YEAR)
    if not op_mask.any():
        print(f"  SKIP  {name}: OP_YEAR {OP_YEAR} not in model_years"); return None
    op_idx = int(np.where(op_mask)[0][0])

    # ── BC daily array ────────────────────────────────────────────────────
    if LOAD_BC_DAILY:
        bc_daily = None
        for _pat in [f"{pfx}_bc_daily_all_years.nc",f"{pfx}_BC_daily_all_years.nc",
                     f"{pfx}_bc_daily_1993_2026.nc","SEAS5_bc_daily_all_years.nc"]:
            bc_daily = _safe_load(_p(_pat))
            if bc_daily is not None: break
        if bc_daily is None:
            print(f"  WARNING {name}: bc_daily not found -- A(D) plume blank")
            bc_daily = np.zeros((n_members, n_years, 182, n_lat, n_lon), np.float32)
    else:
        bc_daily = np.zeros((n_members, n_years, 182, n_lat, n_lon), np.float32)

    # ── GLOB-BASED skill file auto-discovery ──────────────────────────────
    # Scan all .nc files in out_dir and its subdirs once, then match by name.
    # This works regardless of whether files are named:
    #   hitrate_onset_cal.nc  OR  hitrate_cal_onset.nc  OR  onset_hitrate_cal.nc
    _nc_index = {}   # basename (no .nc) -> full path
    for _root, _dirs, _files in os.walk(out_dir):
        for _f in _files:
            if _f.endswith(".nc"):
                _key = _f[:-3].lower()   # strip .nc, lowercase
                _nc_index[_key] = os.path.join(_root, _f)

    def _find_skill(metric, variable, period=None):
        """
        Find the skill file by trying all permutations of the three tokens.
        metric  : "hitrate" | "rpss" | "alpha"
        variable: "onset"   | "cessation" | "lgp"
        period  : "cal"     | "val"       | None
        """
        tokens = [metric, variable] + ([period] if period else [])
        # Build all orderings of the tokens
        from itertools import permutations as _perms
        candidates = ["_".join(p) for p in _perms(tokens)]
        for cand in candidates:
            if cand in _nc_index:
                return _safe_load(_nc_index[cand])
        return None

    _missing = []

    def _skill(metric, variable, period=None):
        arr = _find_skill(metric, variable, period)
        if arr is None:
            _missing.append(f"{metric}_{variable}{'_'+period if period else ''}.nc")
            return np.full((n_lat, n_lon), np.nan, np.float32)
        return arr

    def _probs(metric, variable):
        # probs files are not permuted — try fixed patterns
        for cand in [f"{metric}_{variable}", f"{pfx}_{metric}_{variable}"]:
            if cand.lower() in _nc_index:
                return _safe_load(_nc_index[cand.lower()])
        _missing.append(f"{metric}_{variable}.nc")
        return np.full((3, n_years, n_lat, n_lon), 1/3, np.float32)

    probs_damped = {
        "onset"    : _probs("probs_damped", "onset"),
        "cessation": _probs("probs_damped", "cessation"),
        "lgp"      : _probs("probs_damped", "lgp"),
    }
    alpha = {
        "onset"    : _skill("alpha", "onset"),
        "cessation": _skill("alpha", "cessation"),
        "lgp"      : _skill("alpha", "lgp"),
    }
    hitrate_cal = {
        "onset"    : _skill("hitrate", "onset",     "cal"),
        "cessation": _skill("hitrate", "cessation", "cal"),
        "lgp"      : _skill("hitrate", "lgp",       "cal"),
    }
    hitrate_val = {
        "onset"    : _skill("hitrate", "onset",     "val"),
        "cessation": _skill("hitrate", "cessation", "val"),
        "lgp"      : _skill("hitrate", "lgp",       "val"),
    }
    rpss_val = {
        "onset"    : _skill("rpss",    "onset",     "val"),
        "cessation": _skill("rpss",    "cessation", "val"),
        "lgp"      : _skill("rpss",    "lgp",       "val"),
    }

    # ── Duplication check ─────────────────────────────────────────────────
    _sig = tuple(onset_doy[:min(2,n_members), op_idx, 21, 17].round(1))
    for _pn, _pm in MODELS.items():
        _psig = tuple(_pm["onset_doy"][:min(2,_pm["n_members"]),_pm["op_idx"],21,17].round(1))
        if _sig == _psig:
            print(f"  WARNING {name}: data IDENTICAL to {_pn} -- same directory?")

    if _missing:
        print(f"  WARNING {name}: {len(_missing)} skill file(s) not found "
              f"(NaN filled): {_missing[:4]}")
        if len(_nc_index) < 5:
            print(f"  (only {len(_nc_index)} .nc files found in {out_dir} -- "
                  f"did the pipeline finish?)")
    else:
        print(f"  LOADED {name:<22}: {n_members} members  {n_years} years  "
              f"op_idx={op_idx}  HR={np.nanmean(hitrate_cal["onset"][lm]):.3f}")
        # Print short loaded line only if no missing files
        return {
            "onset_doy"    : onset_doy,    "cessation_doy": cess_doy,
            "lgp_days"     : lgp_days,     "bc_daily"     : bc_daily,
            "probs_damped" : probs_damped, "alpha"        : alpha,
            "hitrate_cal"  : hitrate_cal,  "hitrate_val"  : hitrate_val,
            "rpss_val"     : rpss_val,     "op_idx"       : op_idx,
            "n_members"    : n_members,    "color"        : MODEL_COLORS.get(name,"#888888"),
            "model_years"  : model_years,
        }

    # Missing files — still return (with NaN skill) so bulletin can run
    print(f"  LOADED {name:<22}: {n_members} members  {n_years} years  "
          f"op_idx={op_idx}  (skill files missing)")
    return {
        "onset_doy"    : onset_doy,    "cessation_doy": cess_doy,
        "lgp_days"     : lgp_days,     "bc_daily"     : bc_daily,
        "probs_damped" : probs_damped, "alpha"        : alpha,
        "hitrate_cal"  : hitrate_cal,  "hitrate_val"  : hitrate_val,
        "rpss_val"     : rpss_val,     "op_idx"       : op_idx,
        "n_members"    : n_members,    "color"        : MODEL_COLORS.get(name,"#888888"),
        "model_years"  : model_years,
    }


## Stage 3 — Load All Models


In [5]:
print("Loading all model outputs ...")
print("=" * 60)

MODELS = {}
for name, out_dir in MODEL_DIRS.items():
    entry = _load_model(name, out_dir)
    if entry is not None:
        MODELS[name] = entry

print("=" * 60)
print(f"Models loaded: {len(MODELS)} / {len(MODEL_DIRS)}")
print()

if len(MODELS) == 0:
    raise RuntimeError(
        "No model outputs found. Run each model pipeline notebook "
        "first and check MODEL_DIRS paths above.")

import warnings as _w
print(f"  {'Model':<22}  {'Memb':>5}  {'CAL HR':>8}  {'VAL RPSS':>9}  Status")
print("  " + "-" * 62)
for mname, md in MODELS.items():
    with _w.catch_warnings():
        _w.simplefilter("ignore", RuntimeWarning)
        hr   = float(np.nanmean(md["hitrate_cal"]["onset"][lm]))
        rpss = float(np.nanmean(md["rpss_val"]["onset"][lm]))
    hr_ok = not np.isnan(hr)
    if not hr_ok:
        status = "no skill files"
        hr_s = "  ---  "; rpss_s = "  ---  "
    elif hr > 1/3:
        status = f"PASS  ({hr:.3f} > 0.333)"
        hr_s = f"{hr:.3f}"; rpss_s = f"{rpss:+.3f}"
    else:
        status = f"low   ({hr:.3f} <= 0.333)"
        hr_s = f"{hr:.3f}"; rpss_s = f"{rpss:+.3f}"
    print(f"  {mname:<22}  {md['n_members']:>5}  {hr_s:>8}  {rpss_s:>9}  {status}")


Loading all model outputs ...
  SKIP  ECMWF SEAS5: onset_doy file missing in C:\Users\yonas\Documents\ILRI\onset-kenya\outputs_seas5_v3


C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\2898727244.py:34: FutureWarning: In a future version, xarray will not decode the variable '__xarray_dataarray_variable__' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds  = xr.open_dataset(path)


  LOADED UKMO GloSea6          : 2 members  34 years  op_idx=33  shape=(2, 34, 42, 34)


C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\2898727244.py:34: FutureWarning: In a future version, xarray will not decode the variable '__xarray_dataarray_variable__' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds  = xr.open_dataset(path)


  LOADED Meteo-France Sys8     : 31 members  33 years  op_idx=32  shape=(31, 33, 42, 34)


...

C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\2898727244.py:34: FutureWarning: In a future version, xarray will not decode the variable '__xarray_dataarray_variable__' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds  = xr.open_dataset(path)


  LOADED ECCC CanSIPS          : 20 members  33 years  op_idx=32  shape=(20, 33, 42, 34)


C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\2898727244.py:34: FutureWarning: In a future version, xarray will not decode the variable '__xarray_dataarray_variable__' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds  = xr.open_dataset(path)


  LOADED BOM ACCESS-S2         : 3 members  27 years  op_idx=26  shape=(3, 27, 42, 34)
Models loaded: 7 / 8

  Model                   Members    Onset HR      RPSS  Status
  -----------------------------------------------------------------
  UKMO GloSea6                  2         nan       nan  no skill
  Meteo-France Sys8            31         nan       nan  no skill
  DWD GCFS2.1                  30         nan       nan  no skill
  CMCC-SPS4                    30         nan       nan  no skill
  NCEP CFSv2                    4         nan       nan  no skill
  ECCC CanSIPS                 20         nan       nan  no skill
  BOM ACCESS-S2                 3         nan       nan  no skill


C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\320868189.py:24: RuntimeWarning: Mean of empty slice
  hr   = float(np.nanmean(md_entry['hitrate_cal']['onset'][lm]))
C:\Users\yonas\AppData\Local\Temp\ipykernel_38436\320868189.py:25: RuntimeWarning: Mean of empty slice
  rpss = float(np.nanmean(md_entry['rpss_val']['onset'][lm]))


## Stage 4 — Validate Array Shapes and Consistency

Check that all loaded arrays are consistent with the shared grid.


In [ ]:
print('Validating array shapes ...')
issues = []
for name, md_entry in MODELS.items():
    od = md_entry['onset_doy']
    pd = md_entry['probs_damped']['onset']

    # Onset/cess/lgp must have same shape
    for arr_name in ['cessation_doy', 'lgp_days']:
        if md_entry[arr_name].shape != od.shape:
            issues.append(
                f'{name}: {arr_name} shape {md_entry[arr_name].shape} '
                f'!= onset {od.shape}')

    # Spatial dims must match shared grid
    if od.shape[-2:] != (n_lat, n_lon):
        issues.append(
            f'{name}: spatial dims {od.shape[-2:]} != '
            f'shared grid ({n_lat},{n_lon})')

    # op_idx must be within bounds
    if md_entry['op_idx'] >= od.shape[1]:
        issues.append(
            f'{name}: op_idx {md_entry["op_idx"]} >= '
            f'n_years {od.shape[1]}')

    # probs must have 3 categories
    if pd.shape[0] != 3:
        issues.append(f'{name}: probs dim 0 = {pd.shape[0]}, expected 3')

    print(f'  {name:<22}: onset {od.shape}  probs {pd.shape}  '
          f'op_idx={md_entry["op_idx"]}')

if issues:
    print('\nISSUES FOUND:')
    for iss in issues: print(f'  !! {iss}')
    raise ValueError(f'{len(issues)} shape/consistency issues -- fix before proceeding')
else:
    print('\nAll array shapes consistent ✓')


## Stage 5 — Set Multi-Model Bulletin Metadata

These variables override the defaults in `bulletin_multimodel_v1.py`.
Adjust season strings and init date as needed.


In [ ]:
# These are read by bulletin_multimodel_v1.py via  if 'X' not in globals()
MM_INIT_DATE = 'February 2026'
MM_VALID_STR = 'March - April - May 2026'
MM_RULESET   = (
    'Dunning et al. (2016) daily  |  CHIRPS 0.25 deg  |  '
    'QM-BC per model  |  alpha* pooling'
)

# Always set to today
import datetime as _dt
MM_ISSUED_STR = _dt.date.today().strftime('%d %B %Y')

print(f'  MM_INIT_DATE : {MM_INIT_DATE}')
print(f'  MM_VALID_STR : {MM_VALID_STR}')
print(f'  MM_ISSUED_STR: {MM_ISSUED_STR}')
print(f'  Models       : {list(MODELS.keys())}')


## Stage 6 — Domain-Mean Preview (sanity check before bulletin run)

Print each model's 2026 domain-mean onset/cessation/LGP P50 and
BN/NN/AN probabilities before generating the full bulletin.


In [ ]:
print('2026 Operational Forecast -- Domain-Mean Preview')
print('=' * 75)
print(f'  {"Model":<22}  Onset P50   Cess P50   LGP P50  '
      f'P_BN  P_NN  P_AN')
print('  ' + '-' * 70)

for name, md_entry in MODELS.items():
    oi      = md_entry['op_idx']
    on_vals = md_entry['onset_doy'][:, oi, :, :]
    cs_vals = md_entry['cessation_doy'][:, oi, :, :]
    lg_vals = md_entry['lgp_days'][:, oi, :, :]

    # Domain mean (land pixels)
    on_med = float(np.nanmedian(on_vals[:, lm]))
    cs_med = float(np.nanmedian(cs_vals[:, lm]))
    lg_med = float(np.nanmedian(lg_vals[:, lm]))

    # Domain mean probs
    p = md_entry['probs_damped']['onset'][:, oi, :, :]
    pm = np.nanmean(p[:, lm], axis=1)

    # DOY to date
    def _d(doy):
        try:
            return (datetime.date(OP_YEAR, 1, 1) +
                    datetime.timedelta(days=int(round(doy))-1)
                    ).strftime('%d %b')
        except: return '???'

    print(f'  {name:<22}  '
          f'DOY {on_med:5.0f} ({_d(on_med)})  '
          f'DOY {cs_med:5.0f} ({_d(cs_med)})  '
          f'{lg_med:5.0f}d  '
          f'{pm[0]:.2f}  {pm[1]:.2f}  {pm[2]:.2f}')

print()
c_on = float(np.nanmean(chirps_clim['onset'][lm]))
c_cs = float(np.nanmean(chirps_clim['cessation'][lm]))
c_lg = float(np.nanmean(chirps_clim['lgp'][lm]))
print(f'  {"CHIRPS CAL mean":<22}  '
      f'DOY {c_on:5.0f} ({_d(c_on)})  '
      f'DOY {c_cs:5.0f} ({_d(c_cs)})  {c_lg:5.0f}d')


## Stage 7 — Execute Multi-Model Bulletin Generator

Exec-s `bulletin_multimodel_v1.py` which generates one PNG + PDF
bulletin per site combining all loaded models.

> **If the script file is not in the same directory as this notebook,**
> set `MM_SCRIPT_PATH` manually in the cell below.


In [ ]:
# ── Find bulletin_multimodel_v1.py ───────────────────────────────────────
MM_SCRIPT_PATH = None
_search = [
    os.path.join(BASE_DIR, 'bulletin_multimodel_v1.py'),
    os.path.join(os.getcwd(), 'bulletin_multimodel_v1.py'),
]
import glob as _glob
for _p in _search:
    if os.path.exists(_p):
        MM_SCRIPT_PATH = _p; break
if not MM_SCRIPT_PATH:
    _hits = _glob.glob(
        os.path.join(BASE_DIR, '**', 'bulletin_multimodel_v1.py'),
        recursive=True)
    if _hits: MM_SCRIPT_PATH = _hits[0]

if not MM_SCRIPT_PATH or not os.path.exists(MM_SCRIPT_PATH):
    raise FileNotFoundError(
        'bulletin_multimodel_v1.py not found.\n'
        f'Searched: {_search}\n'
        'Fix: set MM_SCRIPT_PATH = r"C:\\...\\bulletin_multimodel_v1.py"')

print(f'Found: {MM_SCRIPT_PATH}')
print(f'Models: {list(MODELS.keys())}')
print(f'Sites : {len(SITES)}')
print()

with open(MM_SCRIPT_PATH, encoding='utf-8') as _f:
    _mm_src = _f.read()

# Cut before the BATCH RUN section so we can control execution
_mm_cut = len(_mm_src)
for _marker in [
    '# BATCH RUN',
    'print(f"\\nGenerating {len(SITES)}',
    '\nfor s in SITES:',
]:
    _pos = _mm_src.find(_marker)
    if _pos > 0: _mm_cut = _pos; break

# Execute function definitions only
exec(compile(_mm_src[:_mm_cut], MM_SCRIPT_PATH, 'exec'), globals())

# ── Matplotlib compatibility patch ────────────────────────────────────────
# Newer matplotlib (>=3.9) rejects transform= on axhline/axvline.
# Monkey-patch both methods to silently drop the transform kwarg,
# then re-implement using ax.plot so coordinates stay in axes fraction.
import matplotlib.axes as _mplax
import functools as _ft

def _make_safe_axline(orig_method, is_vertical=False):
    """Wrap axhline/axvline to handle transform= kwarg gracefully."""
    @_ft.wraps(orig_method)
    def _safe(*args, **kwargs):
        transform = kwargs.pop("transform", None)
        clip_on   = kwargs.pop("clip_on",   None)
        zorder    = kwargs.pop("zorder",    None)
        if transform is not None:
            # Redraw using ax.plot in the supplied transform coords
            ax   = args[0] if args else None
            val  = args[1] if len(args) > 1 else kwargs.get("x" if is_vertical else "y", 0)
            xmin = kwargs.pop("xmin", 0.0);  xmax = kwargs.pop("xmax", 1.0)
            ymin = kwargs.pop("ymin", 0.0);  ymax = kwargs.pop("ymax", 1.0)
            kw2  = {k: v for k, v in kwargs.items()
                    if k in ("color","lw","ls","alpha","linewidth","linestyle")}
            if zorder   is not None: kw2["zorder"]  = zorder
            if clip_on  is not None: kw2["clip_on"] = clip_on
            if is_vertical:
                ax.plot([val, val], [ymin, ymax], transform=transform, **kw2)
            else:
                ax.plot([xmin, xmax], [val, val], transform=transform, **kw2)
            return None
        # No transform — call original safely
        if clip_on  is not None: kwargs["clip_on"]  = clip_on
        if zorder   is not None: kwargs["zorder"]   = zorder
        return orig_method(*args, **kwargs)
    return _safe

# Only patch if not already patched
if not getattr(_mplax.Axes.axhline, "_mpl_compat_patched", False):
    _mplax.Axes.axhline = _make_safe_axline(_mplax.Axes.axhline, is_vertical=False)
    _mplax.Axes.axhline._mpl_compat_patched = True
if not getattr(_mplax.Axes.axvline, "_mpl_compat_patched", False):
    _mplax.Axes.axvline = _make_safe_axline(_mplax.Axes.axvline, is_vertical=True)
    _mplax.Axes.axvline._mpl_compat_patched = True

# Also patch _hline helper in bulletin namespace
def _hline(ax, y, color, lw=1.0, ls="-", x0=0.0, x1=1.0):
    ax.plot([x0, x1], [y, y], color=color, lw=lw, ls=ls,
            transform=ax.transAxes, clip_on=False, zorder=10)
globals()["_hline"] = _hline

print("  axhline/axvline patched (matplotlib version-safe)")
print(f"  generate_mm_bulletin defined: {'generate_mm_bulletin' in globals()}")


## Stage 8 — Generate Bulletins


In [ ]:
# ── Reload bulletin functions fresh from disk every time ──────────────────
# This guarantees we always run the latest version of bulletin_multimodel_v1.py
# regardless of whether Stage 7 was re-run after a file update.

if "MM_SCRIPT_PATH" not in globals() or not os.path.exists(MM_SCRIPT_PATH):
    # Re-discover script if not already set
    import glob as _g
    _candidates = [
        os.path.join(BASE_DIR, "bulletin_multimodel_v1.py"),
        os.path.join(os.getcwd(), "bulletin_multimodel_v1.py"),
    ] + _g.glob(os.path.join(BASE_DIR, "**", "bulletin_multimodel_v1.py"), recursive=True)
    MM_SCRIPT_PATH = next((p for p in _candidates if os.path.exists(p)), None)
    if not MM_SCRIPT_PATH:
        raise FileNotFoundError(
            "bulletin_multimodel_v1.py not found.\n"
            "Set: MM_SCRIPT_PATH = r'C:\\...\\bulletin_multimodel_v1.py'")

with open(MM_SCRIPT_PATH, encoding="utf-8") as _bf:
    _bsrc = _bf.read()

_cut = len(_bsrc)
for _marker in [
    "# BATCH RUN", "\n# BATCH RUN",
    "print(f\"\\nGenerating {len(SITES)}",
    "\nfor s in SITES:",
]:
    _pos = _bsrc.find(_marker)
    if _pos > 0: _cut = _pos; break

exec(compile(_bsrc[:_cut], MM_SCRIPT_PATH, "exec"), globals())

# ── Force-patch ALL transform-using line helpers (matplotlib compat) ───────
def _hline(ax, y, color, lw=1.0, ls="-", x0=0.0, x1=1.0):
    """Horizontal line in axes-fraction coords. Uses ax.plot (axhline rejects transform=)."""
    ax.plot([x0, x1], [y, y], color=color, lw=lw, ls=ls,
            transform=ax.transAxes, clip_on=False, zorder=10)

def _vline(ax, x, color, lw=1.0, ls="-", y0=0.0, y1=1.0):
    """Vertical line in axes-fraction coords. Uses ax.plot (axvline rejects transform=)."""
    ax.plot([x, x], [y0, y1], color=color, lw=lw, ls=ls,
            transform=ax.transAxes, clip_on=False, zorder=10)

globals()["_hline"] = _hline
globals()["_vline"] = _vline

# Patch axvline calls inside _draw_consensus — the function uses axvline directly
# Monkey-patch ax.axvline at the class level would be too broad.
# Instead, re-define _draw_consensus with the fix inline:
_orig_draw_consensus = globals().get("_draw_consensus")
if _orig_draw_consensus is not None:
    import types, inspect, re as _re

    def _draw_consensus(ax, consensus):
        """Wrapper that neutralises axvline(transform=) inside _draw_consensus."""
        # Temporarily patch the Axes class to convert axvline+transform → ax.plot
        import matplotlib.axes._axes as _mpl_axes
        _orig_axvline = _mpl_axes.Axes.axvline

        def _safe_axvline(self, x=0, ymin=0, ymax=1, **kw):
            kw.pop("transform", None)   # strip transform — generate our own
            kw.pop("clip_on", None)
            self.plot([x, x], [ymin, ymax], transform=self.transAxes,
                      clip_on=False, **kw)

        _mpl_axes.Axes.axvline = _safe_axvline
        try:
            _orig_draw_consensus(ax, consensus)
        finally:
            _mpl_axes.Axes.axvline = _orig_axvline   # restore

    globals()["_draw_consensus"] = _draw_consensus

print(f"  Bulletin reloaded from: {MM_SCRIPT_PATH}")
print(f"  _hline/_vline patched (matplotlib-version-safe)")
print(f"  generate_mm_bulletin defined: {'generate_mm_bulletin' in globals()}")
print()

# ── Generate bulletins ─────────────────────────────────────────────────────
print(f"Generating {len(SITES)} multi-model bulletins ...")
print("=" * 70)

mm_failed = []
for s in SITES:
    try:
        png = generate_mm_bulletin(s["site_name"], s["lat"], s["lon"])
        try:
            from reportlab.lib.pagesizes import A3
            from reportlab.lib.units    import mm
            from reportlab.pdfgen       import canvas as _rl
            from PIL                    import Image as _PIL
            _PIL.MAX_IMAGE_PIXELS = None
            pw, ph = A3; mg = 10 * mm
            img    = _PIL.open(png)
            iw, ih = img.size
            scale  = min((pw-2*mg)/(iw/150*72), (ph-2*mg)/(ih/150*72))
            dw, dh = iw/150*72*scale, ih/150*72*scale
            xo     = mg + ((pw-2*mg)-dw)/2
            yo     = mg + ((ph-2*mg)-dh)/2
            pdf    = png.replace(".png", ".pdf")
            c = _rl.Canvas(pdf, pagesize=(pw, ph))
            c.setTitle(f"Multi-Model Bulletin MAM {OP_YEAR}")
            c.setAuthor("ICPAC / ILRI Climate Services")
            c.drawImage(png, xo, yo, width=dw, height=dh,
                        preserveAspectRatio=True, mask="auto")
            c.save()
            print(f"    PDF saved  ->  {pdf}")
        except Exception as _epdf:
            print(f"    PDF skipped: {_epdf}")
    except Exception as _e:
        import traceback
        print(f"  ERROR  {s['site_name']}: {_e}")
        traceback.print_exc()
        mm_failed.append(s["site_name"])

print("=" * 70)
print(f"Done: {len(SITES)-len(mm_failed)}/{len(SITES)} multi-model bulletins")
if mm_failed:
    print(f"Failed: {mm_failed}")
print(f"Output: {BULLETIN_DIR}")


## Summary

The multi-model bulletin generator has completed.
Each site bulletin contains:

| Panel | Content |
|---|---|
| Header | Site name, all model names, issued date |
| A(D) plume | All model median A(D) curves vs CHIRPS C(d) |
| Timing table | One row per model: onset/cessation/LGP median + spread |
| Grouped bar charts | BN/NN/AN per model per variable |
| Consensus | Equal-weight MMM + skill-screened MMM side by side |
| Agreement heatmap | 3 variables x N models colour grid |
| Risk ranges | 6 risk metrics: min/median/max across models |
| Skill table | HR, RPSS, alpha*, skill screen pass/fail per model |

**Consensus method:** Equal-weight multi-model mean (primary) +
skill-screened equal-weight (secondary, models with HR > 0.333 only).
Full skill-weighting deferred until validation period reaches 15+ years.
